# Campagne S0 — 7 niveaux PWM

Donnees : `data/S0_pwm*.csv` (18 runs). Fits exponentiels sur chaque montee, modeles K(PWM) et tau(PWM) valides par leave-one-out.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import glob

CPR = 960

def step(t, K, tau):
    return K*(1 - np.exp(-t/tau))

In [ ]:
# boucle : tous les fichiers, toutes les montees
DOSSIER = "../data/S0_pwm*.csv"   # adapter si les csv sont ailleurs

rows = []
for f in sorted(glob.glob(DOSSIER)):
    df = pd.read_csv(f)
    level = int(f.split("pwm")[1][:3])
    df["t"] = df.t_ms/1000
    df["rev"] = -df["count"]/CPR
    df["rpm"] = df.rev.diff()/df.t.diff()*60
    for k in range(5):
        m = df[(df.t >= k*4) & (df.t < k*4+2)].dropna(subset=["rpm"])
        (K, tau), _ = curve_fit(step, m.t - k*4, m.rpm, p0=[300, 0.1], maxfev=5000)
        rows.append((level, f.split("_r")[1][0], k+1, K, tau*1000))

res = pd.DataFrame(rows, columns=["pwm", "rep", "cycle", "K", "tau_ms"])
res.groupby("pwm")[["K", "tau_ms"]].agg(["mean", "std"]).round(2)

In [ ]:
# points moyens par niveau
pwm_lv = np.array(sorted(res.pwm.unique()), dtype=float)
K_lv   = np.array([res[res.pwm==l].K.mean() for l in pwm_lv])
tau_lv = np.array([res[res.pwm==l].tau_ms.mean() for l in pwm_lv])

In [ ]:
# course de formes pour K(PWM), validation leave-one-out
def f_expo(p, Kmax, c):  return Kmax*(1 - np.exp(-p/c))
def f_hyper(p, Kmax, c): return Kmax*p/(c + p)
def f_puiss(p, a, b):    return a*p**b

for nom, f, p0 in [("expo saturante", f_expo, [380, 60]),
                   ("hyperbole", f_hyper, [450, 60]),
                   ("puissance", f_puiss, [40, 0.4])]:
    popt, _ = curve_fit(f, pwm_lv, K_lv, p0=p0, maxfev=10000)
    rmse_fit = np.sqrt(np.mean((K_lv - f(pwm_lv, *popt))**2))
    errs = []
    for i in range(len(pwm_lv)):
        g = np.ones(len(pwm_lv), bool); g[i] = False
        po, _ = curve_fit(f, pwm_lv[g], K_lv[g], p0=p0, maxfev=10000)
        errs.append(K_lv[i] - f(pwm_lv[i], *po))
    print(f"{nom:15s} fit={rmse_fit:5.2f} rpm | prediction={np.sqrt(np.mean(np.array(errs)**2)):5.2f} rpm")

In [ ]:
# gagnant K + formule + graphe
poptK, _ = curve_fit(f_expo, pwm_lv, K_lv, p0=[380, 60])
print(f"K(PWM) = {poptK[0]:.1f} * (1 - exp(-PWM/{poptK[1]:.1f}))  [tr/min]")
print(f"loi inverse : PWM = -{poptK[1]:.1f} * ln(1 - K/{poptK[0]:.1f})")
print(f"test aveugle PWM 115 : K = {f_expo(115, *poptK):.1f} tr/min")

pp = np.linspace(40, 260, 200)
plt.figure(figsize=(8, 4.5))
plt.errorbar(pwm_lv, K_lv, yerr=1, fmt="o", color="crimson", capsize=4, label="mesures")
plt.plot(pp, f_expo(pp, *poptK), "k", lw=1.8, label="expo saturante")
plt.plot([0, 255], [0, 255*336.5/150], "--", color="gray", alpha=.6, label="si lineaire")
plt.xlabel("PWM"); plt.ylabel("K (tr/min)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()

In [ ]:
# meme course pour tau(PWM)
def g_expo(p, tmin, A, c): return tmin + A*np.exp(-p/c)
def g_puiss(p, a, b):      return a*p**b
def g_hyper(p, a, c):      return a/(p + c)

for nom, g, p0 in [("expo decroissante", g_expo, [40, 400, 80]),
                   ("puissance", g_puiss, [5000, -0.8]),
                   ("hyperbole", g_hyper, [20000, 30])]:
    popt, _ = curve_fit(g, pwm_lv, tau_lv, p0=p0, maxfev=20000)
    rmse_fit = np.sqrt(np.mean((tau_lv - g(pwm_lv, *popt))**2))
    errs = []
    for i in range(len(pwm_lv)):
        gd = np.ones(len(pwm_lv), bool); gd[i] = False
        po, _ = curve_fit(g, pwm_lv[gd], tau_lv[gd], p0=p0, maxfev=20000)
        errs.append(tau_lv[i] - g(pwm_lv[i], *po))
    print(f"{nom:18s} fit={rmse_fit:5.2f} ms | prediction={np.sqrt(np.mean(np.array(errs)**2)):5.2f} ms")

In [ ]:
# gagnant tau + formule + graphe
poptT, _ = curve_fit(g_expo, pwm_lv, tau_lv, p0=[40, 400, 80], maxfev=20000)
print(f"tau(PWM) = {poptT[0]:.1f} + {poptT[1]:.1f} * exp(-PWM/{poptT[2]:.1f})  [ms]")
print(f"test aveugle PWM 115 : tau = {g_expo(115, *poptT):.1f} ms")

pp = np.linspace(50, 260, 200)
plt.figure(figsize=(8, 4.5))
plt.errorbar(pwm_lv, tau_lv, yerr=2, fmt="s", color="#1f77b4", capsize=4, label="mesures")
plt.plot(pp, g_expo(pp, *poptT), "k", lw=1.8, label="expo decroissante")
plt.xlabel("PWM"); plt.ylabel("tau (ms)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()

## Conclusions

- K(PWM) = 369.2 (1 - e^(-PWM/60.6)) tr/min — prediction LOO ~3.9 tr/min
- tau(PWM) = 29.3 + 383.6 e^(-PWM/86.5) ms — prediction LOO ~1.2 ms
- Actionneur fortement non lineaire (drive-coast) : structure Hammerstein ou linearisation locale pour la suite
- Comparaison Pololu : +12% sur la vitesse a vide, a elucider (VIN, tolerance, ticks/tour)
- Promesses du test aveugle PWM 115 : K = 314 tr/min, tau = 131 ms